# Table 1 — Summary Statistics

Builds a single **Table 1** with four panels (Demographics, Self-Assessment, Knowledge Outcomes, Duration)  
and a companion **Table 2** showing per-question correctness by group.

- Continuous / ordinal variables: **median (Q1–Q3)**, Kruskal-Wallis p
- Categorical variables: **n (%)**, chi-square p (Fisher's exact for 2×2)
- Groups ordered: No Resource → PDF → ChatGPT

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display, HTML

GROUP_ORDER = ["No Resource", "PDF", "ChatGPT"]
GROUP_N     = {"No Resource": 7, "PDF": 5, "ChatGPT": 6}
EDU_ORDER   = [
    "Medical Student",
    "Resident/Fellow",
    "Attending",
    "Advanced Practice Provider",
]

In [2]:
df = pd.read_csv("../data/processed/survey_clean.csv")
qlabels = pd.read_csv("../data/processed/question_labels.csv", index_col="column")

df["group_label"] = pd.Categorical(
    df["group_label"], categories=GROUP_ORDER, ordered=False
)
df["edu_status"] = pd.Categorical(
    df["edu_status"], categories=EDU_ORDER, ordered=True
)

# Per-group sub-frames
groups = {g: df[df["group_label"] == g].reset_index(drop=True) for g in GROUP_ORDER}

print(f"Total N = {len(df)}")
print(df["group_label"].value_counts().reindex(GROUP_ORDER))

Total N = 18
group_label
No Resource    7
PDF            5
ChatGPT        6
Name: count, dtype: int64


## Helper functions

In [3]:
def fmt_p(p: float) -> str:
    """Format a p-value to 3 decimals; cap at p < 0.001."""
    if pd.isna(p):
        return "—"
    if p < 0.001:
        return "< 0.001"
    return f"{p:.3f}"


def summarise_continuous(series: pd.Series) -> str:
    """Return 'median (Q1–Q3)' string, 1 decimal place."""
    s = series.dropna()
    if len(s) == 0:
        return "—"
    med = s.median()
    q1  = s.quantile(0.25)
    q3  = s.quantile(0.75)
    return f"{med:.1f} ({q1:.1f}\u2013{q3:.1f})"


def summarise_categorical(
    series: pd.Series,
    n_total: int,
    category: object,
) -> str:
    """Return 'n (%)' string for a single category."""
    n = int((series == category).sum())
    pct = 100.0 * n / n_total if n_total > 0 else 0.0
    return f"{n} ({pct:.1f}%)"


def kruskal_p(*group_series) -> float:
    """Kruskal-Wallis p-value; NaN if any group has < 2 observations."""
    cleaned = [s.dropna() for s in group_series]
    if any(len(s) < 2 for s in cleaned):
        return np.nan
    try:
        _, p = stats.kruskal(*cleaned)
        return float(p)
    except Exception:
        return np.nan


def chi2_p(contingency_array: np.ndarray) -> float:
    """Chi-square p-value for an N×K contingency table."""
    try:
        _, p, _, _ = stats.chi2_contingency(contingency_array)
        return float(p)
    except Exception:
        return np.nan


def fisher2x2_p(contingency_2x2: np.ndarray) -> float:
    """Fisher's exact p-value for a 2×2 table."""
    try:
        _, p = stats.fisher_exact(contingency_2x2)
        return float(p)
    except Exception:
        return np.nan


print("Helper functions defined.")

Helper functions defined.


---
## Panel A — Demographics

Continuous and ordinal demographic variables (e.g., age if present) are reported as
median (Q1–Q3) and tested with the **Kruskal-Wallis** rank-sum test — a non-parametric
alternative to one-way ANOVA, selected a priori because group sizes (n = 5–7) are too
small to verify parametric assumptions.

Categorical variables (education level, specialty) are reported as n (%) but marked
**NT (not tested)** in the p-value column. With only 18 participants spread across 3
groups, most cells in the contingency tables have expected counts < 5, which violates
the assumptions of both Pearson's chi-square and Fisher's exact test. Reporting a
p-value from these tests would be numerically unreliable.

In [4]:
rows_a = []
N = len(df)

# ── Education level ────────────────────────────────────────────────────────
rows_a.append({
    "Variable": "Education level, n (%)",
    "Overall (N=18)": "",
    "No Resource (n=7)": "",
    "PDF (n=5)": "",
    "ChatGPT (n=6)": "",
    "p-value": "",
})

for i, edu in enumerate(EDU_ORDER):
    row = {
        "Variable": f"  {edu}",
        "Overall (N=18)": summarise_categorical(df["edu_status"], N, edu),
        "p-value": "NT",
    }
    for g in GROUP_ORDER:
        col_key = f"{g} (n={GROUP_N[g]})"
        row[col_key] = summarise_categorical(groups[g]["edu_status"], len(groups[g]), edu)
    rows_a.append(row)

# ── Specialty ──────────────────────────────────────────────────────────────
rows_a.append({
    "Variable": "Specialty, n (%)",
    "Overall (N=18)": "",
    "No Resource (n=7)": "",
    "PDF (n=5)": "",
    "ChatGPT (n=6)": "",
    "p-value": "",
})

KEEP_SPECS = {
    "Emergency Medicine",
    "Pediatric Emergency Medicine",
    "Pediatric Medicine",
    "Family Medicine",
}

def collapse_specialty(s):
    if pd.isna(s) or str(s).strip() == "":
        return "Not specified"
    return s if s in KEEP_SPECS else "Other"

df["specialty_col"] = df["specialty"].apply(collapse_specialty)
for g in GROUP_ORDER:
    groups[g] = groups[g].copy()
    groups[g]["specialty_col"] = groups[g]["specialty"].apply(collapse_specialty)

# Build ordered spec list using only values that exist in the data
SPEC_ORDER = [
    "Emergency Medicine",
    "Pediatric Emergency Medicine",
    "Pediatric Medicine",
    "Family Medicine",
    "Not specified",
    "Other",
]
present_specs = [s for s in SPEC_ORDER if (df["specialty_col"] == s).any()]

for i, spec in enumerate(present_specs):
    row = {
        "Variable": f"  {spec}",
        "Overall (N=18)": summarise_categorical(df["specialty_col"], N, spec),
        "p-value": "NT",
    }
    for g in GROUP_ORDER:
        col_key = f"{g} (n={GROUP_N[g]})"
        row[col_key] = summarise_categorical(groups[g]["specialty_col"], len(groups[g]), spec)
    rows_a.append(row)

COL_ORDER = [
    "Variable", "Overall (N=18)",
    "No Resource (n=7)", "PDF (n=5)", "ChatGPT (n=6)",
    "p-value",
]
panel_a = pd.DataFrame(rows_a, columns=COL_ORDER).fillna("")
display(panel_a)

,Variable,Overall (N=18),No Resource (n=7),PDF (n=5),ChatGPT (n=6),p-value
0,"Education level, n (%)",,,,,
1,Medical Student,4 (22.2%),1 (14.3%),1 (20.0%),2 (33.3%),NT
2,Resident/Fellow,5 (27.8%),3 (42.9%),1 (20.0%),1 (16.7%),NT
3,Attending,6 (33.3%),3 (42.9%),2 (40.0%),1 (16.7%),NT
4,Advanced Practice Provider,3 (16.7%),0 (0.0%),1 (20.0%),2 (33.3%),NT
5,"Specialty, n (%)",,,,,
6,Emergency Medicine,5 (27.8%),2 (28.6%),1 (20.0%),2 (33.3%),NT
7,Pediatric Emergency Medicine,2 (11.1%),2 (28.6%),0 (0.0%),0 (0.0%),NT
8,Pediatric Medicine,3 (16.7%),1 (14.3%),2 (40.0%),0 (0.0%),NT
9,Family Medicine,1 (5.6%),1 (14.3%),0 (0.0%),0 (0.0%),NT


---
## Panel B — Self-Assessment (pre-case ratings, 0–10 scale)

Participants rated their pre-case self-confidence on a 0–10 Likert scale.
All four variables (`self_knowledge_tdi`, `self_confidence_avulsion`,
`self_confidence_fracture`, `self_confidence_mean`) are ordinal, so
**Kruskal-Wallis** is used for between-group comparisons (same rationale as Panel A).

In [5]:
SELF_VARS = {
    "self_knowledge_tdi":       "TDI knowledge (0\u201310)",
    "self_confidence_avulsion": "Avulsion confidence (0\u201310)",
    "self_confidence_fracture": "Fracture confidence (0\u201310)",
    "self_confidence_mean":     "Average self-assessment (0\u201310)",
}

rows_b = [{
    "Variable": "Self-assessment, median (IQR)",
    "Overall (N=18)": "", "No Resource (n=7)": "",
    "PDF (n=5)": "", "ChatGPT (n=6)": "", "p-value": "",
}]

for col, label in SELF_VARS.items():
    p = kruskal_p(*[groups[g][col] for g in GROUP_ORDER])
    row = {
        "Variable": f"  {label}",
        "Overall (N=18)": summarise_continuous(df[col]),
        "p-value": fmt_p(p),
    }
    for g in GROUP_ORDER:
        row[f"{g} (n={GROUP_N[g]})"] = summarise_continuous(groups[g][col])
    rows_b.append(row)

panel_b = pd.DataFrame(rows_b, columns=COL_ORDER).fillna("")
display(panel_b)

,Variable,Overall (N=18),No Resource (n=7),PDF (n=5),ChatGPT (n=6),p-value
0,"Self-assessment, median (IQR)",,,,,
1,TDI knowledge (0–10),3.0 (2.0–6.8),5.0 (1.5–7.0),3.0 (3.0–7.0),2.5 (1.2–5.2),0.639
2,Avulsion confidence (0–10),5.5 (2.0–7.0),6.0 (3.0–7.5),3.0 (3.0–9.0),4.0 (2.0–6.0),0.620
3,Fracture confidence (0–10),5.0 (2.2–6.0),6.0 (2.5–7.0),3.0 (3.0–9.0),4.5 (2.2–6.0),0.759
4,Average self-assessment (0–10),4.2 (1.8–6.3),6.0 (2.3–7.0),3.0 (3.0–8.3),3.3 (1.8–5.8),0.761


---
## Panel C — Knowledge Assessment Outcomes

Count outcomes (`n_correct`, `n_incorrect`, `n_not_sure`) and the derived percentage
(`pct_correct_of_attempted`) are all bounded integers or ratios — **Kruskal-Wallis**
is appropriate here for the same reasons as Panels A and B.

In [6]:
OUTCOME_VARS = {
    "n_correct":               "Total correct (out of 12)",
    "n_incorrect":             "Total incorrect",
    "n_not_sure":              "Total \u2018not sure\u2019",
    "pct_correct_of_attempted": "Percent correct of attempted (%)",
}

rows_c = [{
    "Variable": "Knowledge outcomes, median (IQR)",
    "Overall (N=18)": "", "No Resource (n=7)": "",
    "PDF (n=5)": "", "ChatGPT (n=6)": "", "p-value": "",
}]

for col, label in OUTCOME_VARS.items():
    p = kruskal_p(*[groups[g][col] for g in GROUP_ORDER])
    row = {
        "Variable": f"  {label}",
        "Overall (N=18)": summarise_continuous(df[col]),
        "p-value": fmt_p(p),
    }
    for g in GROUP_ORDER:
        row[f"{g} (n={GROUP_N[g]})"] = summarise_continuous(groups[g][col])
    rows_c.append(row)

panel_c = pd.DataFrame(rows_c, columns=COL_ORDER).fillna("")
display(panel_c)

,Variable,Overall (N=18),No Resource (n=7),PDF (n=5),ChatGPT (n=6),p-value
0,"Knowledge outcomes, median (IQR)",,,,,
1,Total correct (out of 12),8.0 (7.0–9.8),8.0 (6.5–8.5),9.0 (7.0–10.0),8.5 (7.2–10.5),0.577
2,Total incorrect,3.5 (2.2–4.0),4.0 (3.0–4.0),3.0 (2.0–5.0),3.5 (1.5–4.0),0.890
3,Total ‘not sure’,0.0 (0.0–0.0),0.0 (0.0–1.5),0.0 (0.0–0.0),0.0 (0.0–0.0),0.405
4,Percent correct of attempted (%),66.7 (62.8–81.2),66.7 (64.6–70.8),75.0 (58.3–83.3),70.8 (64.4–87.5),0.726


---
## Panel D — Survey Duration

`duration_min` (converted from seconds) measures total time spent. It is continuous
but right-skewed in small samples, so **Kruskal-Wallis** is again preferred over a
parametric F-test.

In [7]:
p_dur = kruskal_p(*[groups[g]["duration_min"] for g in GROUP_ORDER])

dur_row = {
    "Variable": "  Completion time (min)",
    "Overall (N=18)": summarise_continuous(df["duration_min"]),
    "p-value": fmt_p(p_dur),
}
for g in GROUP_ORDER:
    dur_row[f"{g} (n={GROUP_N[g]})"] = summarise_continuous(groups[g]["duration_min"])

rows_d = [
    {
        "Variable": "Survey duration, median (IQR)",
        "Overall (N=18)": "", "No Resource (n=7)": "",
        "PDF (n=5)": "", "ChatGPT (n=6)": "", "p-value": "",
    },
    dur_row,
]

panel_d = pd.DataFrame(rows_d, columns=COL_ORDER).fillna("")
display(panel_d)

,Variable,Overall (N=18),No Resource (n=7),PDF (n=5),ChatGPT (n=6),p-value
0,"Survey duration, median (IQR)",,,,,
1,Completion time (min),8.3 (6.0–14.7),5.9 (5.1–7.0),14.8 (10.4–17.8),10.2 (7.0–13.7),0.145


---
## Assemble Table 1

In [8]:
def section_header(title: str) -> pd.DataFrame:
    return pd.DataFrame([{c: "" for c in COL_ORDER} | {"Variable": title}])


sep = pd.DataFrame([{c: "" for c in COL_ORDER}])

table1 = pd.concat(
    [
        section_header("Panel A \u2014 Demographics"),
        panel_a,
        sep,
        section_header("Panel B \u2014 Self-Assessment (pre-case, 0\u201310 scale)"),
        panel_b,
        sep,
        section_header("Panel C \u2014 Knowledge Assessment Outcomes"),
        panel_c,
        sep,
        section_header("Panel D \u2014 Survey Duration"),
        panel_d,
    ],
    ignore_index=True,
).fillna("")

display(table1)

,Variable,Overall (N=18),No Resource (n=7),PDF (n=5),ChatGPT (n=6),p-value
0,Panel A — Demographics,,,,,
1,"Education level, n (%)",,,,,
2,Medical Student,4 (22.2%),1 (14.3%),1 (20.0%),2 (33.3%),NT
3,Resident/Fellow,5 (27.8%),3 (42.9%),1 (20.0%),1 (16.7%),NT
4,Attending,6 (33.3%),3 (42.9%),2 (40.0%),1 (16.7%),NT
5,Advanced Practice Provider,3 (16.7%),0 (0.0%),1 (20.0%),2 (33.3%),NT
6,"Specialty, n (%)",,,,,
7,Emergency Medicine,5 (27.8%),2 (28.6%),1 (20.0%),2 (33.3%),NT
8,Pediatric Emergency Medicine,2 (11.1%),2 (28.6%),0 (0.0%),0 (0.0%),NT
9,Pediatric Medicine,3 (16.7%),1 (14.3%),2 (40.0%),0 (0.0%),NT


---
## Table 2 — Per-question Correctness by Group (companion table)

Each cell shows *n* correct / *N* (%).

**Statistical test:** Pearson's chi-square applied to the 2×3 (correct/incorrect × group)
contingency table for each question. Chi-square is appropriate here because each question
yields a binary outcome (correct vs. incorrect) compared across three independent groups.
Note that with very small group sizes the expected cell counts may fall below 5 for some
questions; these results should be interpreted with caution.

**Bonferroni correction:** With k = 12 simultaneous chi-square tests (one per question),
we apply a Bonferroni adjustment: `p_corrected = p × 12`, capped at 1.0. This controls
the family-wise error rate at α = 0.05, yielding an adjusted threshold of **α = 0.0042**.
Both raw and Bonferroni-corrected p-values are shown so readers can judge significance
under either criterion.

In [9]:
Q_COLS = [
    "c1_injury_type_correct", "c1_treatment_correct", "c1_antibiotics_correct",
    "c2_injury_type_correct", "c2_treatment_correct", "c2_tf_60min_correct",
    "c2_storage_rank_correct", "c2_antibiotics_correct",
    "c3_injury_type_correct", "c3_treatment_correct",
    "c3_imaging_correct",      "c3_antibiotics_correct",
]

N_COMPARISONS = len(Q_COLS)

Q_COL_ORDER = [
    "Question", "Overall (N=18)",
    "No Resource (n=7)", "PDF (n=5)", "ChatGPT (n=6)",
    "p-value", "p-value (corrected)",
]

q_rows = []
raw_p_values = []

for col in Q_COLS:
    base = col.replace("_correct", "")
    label = qlabels.loc[base, "label"] if base in qlabels.index else col

    n_all = int(df[col].sum())
    N_all = len(df)
    row = {
        "Question": label,
        "Overall (N=18)": f"{n_all}/{N_all} ({100*n_all/N_all:.1f}%)",
    }

    correct_counts   = []
    incorrect_counts = []
    for g in GROUP_ORDER:
        n_g = int(groups[g][col].sum())
        N_g = len(groups[g])
        row[f"{g} (n={GROUP_N[g]})"] = f"{n_g}/{N_g} ({100*n_g/N_g:.1f}%)"
        correct_counts.append(n_g)
        incorrect_counts.append(N_g - n_g)

    ct_2x3 = np.array([correct_counts, incorrect_counts])
    p_raw = chi2_p(ct_2x3)
    raw_p_values.append(p_raw)

    p_corr = min(p_raw * N_COMPARISONS, 1.0) if not pd.isna(p_raw) else np.nan

    row["p-value"] = fmt_p(p_raw)
    row["p-value (corrected)"] = fmt_p(p_corr)

    q_rows.append(row)

per_q_table = pd.DataFrame(q_rows, columns=Q_COL_ORDER)
display(per_q_table)

,Question,Overall (N=18),No Resource (n=7),PDF (n=5),ChatGPT (n=6),p-value,p-value (corrected)
0,C1: Injury type,17/18 (94.4%),7/7 (100.0%),4/5 (80.0%),6/6 (100.0%),0.252,1.000
1,C1: Treatment,17/18 (94.4%),7/7 (100.0%),4/5 (80.0%),6/6 (100.0%),0.252,1.000
2,C1: Antibiotics,16/18 (88.9%),7/7 (100.0%),4/5 (80.0%),5/6 (83.3%),0.481,1.000
3,C2: Injury type,15/18 (83.3%),6/7 (85.7%),5/5 (100.0%),4/6 (66.7%),0.328,1.000
4,C2: Treatment,13/18 (72.2%),5/7 (71.4%),3/5 (60.0%),5/6 (83.3%),0.689,1.000
5,C2: >60 min outside mouth (T/F),16/18 (88.9%),5/7 (71.4%),5/5 (100.0%),6/6 (100.0%),0.171,1.000
6,C2: Storage medium ranking,15/18 (83.3%),6/7 (85.7%),5/5 (100.0%),4/6 (66.7%),0.328,1.000
7,C2: Antibiotic therapy,1/18 (5.6%),0/7 (0.0%),0/5 (0.0%),1/6 (16.7%),0.347,1.000
8,C3: Injury type,12/18 (66.7%),2/7 (28.6%),4/5 (80.0%),6/6 (100.0%),0.019,0.223
9,C3: Treatment,16/18 (88.9%),6/7 (85.7%),4/5 (80.0%),6/6 (100.0%),0.543,1.000


---
## Export

In [10]:
import os

TABLES_DIR = "../tables"
os.makedirs(TABLES_DIR, exist_ok=True)

# ── CSV ────────────────────────────────────────────────────────────────────
table1.to_csv(f"{TABLES_DIR}/table1_summary.csv", index=False)
per_q_table.to_csv(f"{TABLES_DIR}/table2_per_question.csv", index=False)
print("CSV saved.")

# ── HTML ───────────────────────────────────────────────────────────────────
html_style = """
<style>
  table  { border-collapse: collapse; font-family: Arial, sans-serif; font-size: 13px; }
  th, td { border: 1px solid #ccc; padding: 6px 10px; text-align: left; }
  th     { background: #f0f0f0; font-weight: bold; }
  tr:nth-child(even) { background: #fafafa; }
</style>
"""

with open(f"{TABLES_DIR}/table1_summary.html", "w", encoding="utf-8") as fh:
    fh.write(html_style)
    fh.write("<h2>Table 1 &#8212; Summary Statistics</h2>\n")
    fh.write(table1.to_html(index=False, border=0, justify="left"))

with open(f"{TABLES_DIR}/table2_per_question.html", "w", encoding="utf-8") as fh:
    fh.write(html_style)
    fh.write("<h2>Table 2 &#8212; Per-question Correctness by Group</h2>\n")
    fh.write(per_q_table.to_html(index=False, border=0, justify="left"))

print("HTML saved.")

# ── LaTeX ──────────────────────────────────────────────────────────────────
with open(f"{TABLES_DIR}/table1_summary.tex", "w", encoding="utf-8") as fh:
    fh.write(table1.to_latex(index=False, escape=True, longtable=True))

with open(f"{TABLES_DIR}/table2_per_question.tex", "w", encoding="utf-8") as fh:
    fh.write(per_q_table.to_latex(index=False, escape=True, longtable=True))

print("LaTeX saved.")

# ── Excel ──────────────────────────────────────────────────────────────────
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL   = PatternFill("solid", fgColor="D9D9D9")
SECTION_FILL  = PatternFill("solid", fgColor="EEF2FF")
THIN_BORDER   = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin"),
)
LEGEND_FONT   = Font(size=9, italic=True, color="444444")

def _style_sheet(ws, df: pd.DataFrame) -> None:
    """Apply basic formatting to a worksheet already populated by to_excel."""
    last_data_row = len(df) + 1  # +1 for the header row

    for col_idx, col_name in enumerate(df.columns, start=1):
        max_len = max(
            len(str(col_name)),
            df[col_name].astype(str).str.len().max() if len(df) > 0 else 0,
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 4, 50)

    for row in ws.iter_rows(min_row=1, max_row=last_data_row):
        for cell in row:
            cell.border    = THIN_BORDER
            cell.alignment = Alignment(wrap_text=True, vertical="center")
            if cell.row == 1:
                cell.font = Font(bold=True)
                cell.fill = HEADER_FILL

    var_col_idx = 1
    for row in ws.iter_rows(min_row=2, max_row=last_data_row):
        var_cell = row[var_col_idx - 1]
        other_vals = [c.value for c in row[var_col_idx:] if c.value not in (None, "")]
        if var_cell.value and not str(var_cell.value).startswith("  ") and not other_vals:
            for cell in row:
                cell.fill = SECTION_FILL
                cell.font = Font(bold=True)

    ws.freeze_panes = "B2"


NO_BORDER = Border(
    left=Side(style=None), right=Side(style=None),
    top=Side(style=None),  bottom=Side(style=None),
)

def _write_legend(ws, df: pd.DataFrame, legend_text: str) -> None:
    """Write a merged legend block two rows below the data."""
    from openpyxl.utils import get_column_letter

    n_cols = len(df.columns)
    gap_row    = len(df) + 2  # +1 header, +1 blank
    legend_row = gap_row + 1

    for col_idx in range(1, n_cols + 1):
        gap_cell = ws.cell(row=gap_row, column=col_idx)
        gap_cell.border = NO_BORDER

    start_cell = ws.cell(row=legend_row, column=1, value=legend_text)
    start_cell.font = LEGEND_FONT
    start_cell.alignment = Alignment(wrap_text=True, vertical="top")
    start_cell.border = NO_BORDER

    for col_idx in range(2, n_cols + 1):
        ws.cell(row=legend_row, column=col_idx).border = NO_BORDER

    if n_cols > 1:
        end_col_letter = get_column_letter(n_cols)
        ws.merge_cells(f"A{legend_row}:{end_col_letter}{legend_row}")

    ws.row_dimensions[legend_row].height = 80


TABLE1_LEGEND = (
    "Participants (N = 18) were randomly assigned to one of three resource "
    "groups: No Resource (n = 7), PDF reference (n = 5), or ChatGPT (n = 6). "
    "Continuous and ordinal variables are reported as median (Q1\u2013Q3); "
    "categorical variables as n (%). Between-group differences for ordinal "
    "and continuous variables were assessed with the Kruskal-Wallis rank-sum "
    "test, a non-parametric method selected a priori given the small, unequal "
    "group sizes (n = 5\u20137), for which parametric assumptions cannot be "
    "reliably verified. Categorical demographic variables (education level, "
    "specialty) were not tested for between-group differences (NT) because "
    "the sparse contingency tables (N = 18 across 3 groups) produced expected "
    "cell counts < 5 in the majority of cells, violating the assumptions of "
    "both the chi-square test and Fisher\u2019s exact test. "
    "IQR, interquartile range; NT, not tested."
)

TABLE2_LEGEND = (
    "Per-question correctness across three resource groups (No Resource, "
    "n = 7; PDF, n = 5; ChatGPT, n = 6). Each cell shows the number "
    "correct / total (%). P-values were derived from Pearson\u2019s chi-square "
    "test applied to 2 \u00d7 3 (correct/incorrect \u00d7 group) contingency "
    "tables, chosen because each question yields a binary outcome (correct "
    "vs. incorrect) compared across three independent groups. Given the small "
    "sample and the number of simultaneous comparisons (k = 12 questions), "
    "Bonferroni-corrected p-values are provided (p_corrected = p \u00d7 k, "
    "capped at 1.0) to control the family-wise error rate."
)

excel_path = f"{TABLES_DIR}/tables_summary.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    table1.to_excel(writer, sheet_name="Table 1 \u2014 Summary", index=False)
    per_q_table.to_excel(writer, sheet_name="Table 2 \u2014 Per-question", index=False)

    ws1 = writer.sheets["Table 1 \u2014 Summary"]
    ws2 = writer.sheets["Table 2 \u2014 Per-question"]

    _style_sheet(ws1, table1)
    _style_sheet(ws2, per_q_table)

    _write_legend(ws1, table1,      TABLE1_LEGEND)
    _write_legend(ws2, per_q_table, TABLE2_LEGEND)

print("Excel saved.")
print("\nAll outputs written to tables/")

CSV saved.
HTML saved.


LaTeX saved.


Excel saved.

All outputs written to tables/


---
## Table 3 — Spearman Correlations (Self-Confidence vs Knowledge Score)

Spearman's rank-order correlation between `self_confidence_mean` and `n_correct`  
is computed for all 18 participants and separately for each experimental group.

**Why Spearman?** Both variables are ordinal in nature (Likert-based confidence ratings,  
count of correct answers) and normality cannot be assumed with n = 5\u20137 per group.  
Spearman's \u03c1 makes no distributional assumptions and is appropriate for small samples.

**Bonferroni correction:** Four simultaneous comparisons are performed (1 overall + 3 per group),  
so adjusted p-values are computed as p \u00d7 4, capped at 1.0. The Bonferroni-adjusted  
significance threshold is **\u03b1 = 0.0125** (0.05 \u00f7 4).

In [11]:
from scipy.stats import spearmanr

subsets = [
    ("All Participants", df),
    ("No Resource",      df[df["group_label"] == "No Resource"]),
    ("PDF",              df[df["group_label"] == "PDF"]),
    ("ChatGPT",          df[df["group_label"] == "ChatGPT"]),
]
N_CORR_TESTS = len(subsets)

corr_rows = []
for label, sub in subsets:
    valid = sub[["self_confidence_mean", "n_correct"]].dropna()
    n = len(valid)
    if n >= 3:
        rho, pval = spearmanr(valid["self_confidence_mean"], valid["n_correct"])
        p_adj = min(pval * N_CORR_TESTS, 1.0)
        corr_rows.append({
            "Subset":       label,
            "n":            n,
            "rho":          round(rho, 2),
            "p_value":      round(pval, 3) if pval >= 0.001 else "< 0.001",
            "p_bonferroni": round(p_adj, 3) if p_adj >= 0.001 else "< 0.001",
        })
    else:
        corr_rows.append({"Subset": label, "n": n,
                          "rho": None, "p_value": None, "p_bonferroni": None})

table3 = pd.DataFrame(corr_rows)
display(table3)

# ── Export ──────────────────────────────────────────────────────────────────
table3.to_csv(f"{TABLES_DIR}/table3_correlation.csv", index=False)
print("Table 3 CSV saved.")

TABLE3_LEGEND = (
    "Spearman rank-order correlation between average self-rated confidence "
    "(self_confidence_mean, 0\u201310 scale) and total correct responses "
    "(n_correct, 0\u201312) for the full sample and each experimental group. "
    "Bonferroni-adjusted p-values account for 4 simultaneous tests "
    "(adjusted \u03b1 = 0.0125). "
    "Per-group estimates are based on very small subsamples (n = 5\u20137) "
    "and should be interpreted with caution."
)

# Append Table 3 sheet to existing Excel file
excel_path = f"{TABLES_DIR}/tables_summary.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as writer:
    table3.to_excel(writer, sheet_name="Table 3 \u2014 Correlation", index=False)
    ws3 = writer.sheets["Table 3 \u2014 Correlation"]
    _style_sheet(ws3, table3)
    _write_legend(ws3, table3, TABLE3_LEGEND)

print("Table 3 sheet added to Excel.")

,Subset,n,rho,p_value,p_bonferroni
0,All Participants,18,0.35,0.149,0.595
1,No Resource,7,0.93,0.003,0.011
2,PDF,5,0.24,0.701,1.000
3,ChatGPT,6,-0.09,0.866,1.000


Table 3 CSV saved.
Table 3 sheet added to Excel.
